# 41 · Analysis: which industries are growing or shrinking in NYC?

**Rules:** written before this notebook ran. Reads: `fct_industry_employment_annual` (QCEW, NYS DOL + BLS county fills), `fct_industry_hiring_quarterly` (Census QWI), `fct_industry_employment_monthly` (CES, NYS DOL), `fct_industry_gdp` (BEA county GDP), `bridge_industry_job` (OEWS research NY staffing pattern), `dim_industry`, `dim_area`. Changes nothing.

Every number here counts employers in this area (industry data), not people doing a job (S8). The industry is one of the two ways into the job hub (S4), so the last section links each sector to the degree-reachable jobs inside it.

**Questions**
1. Which NYC sectors added or lost jobs 2019 → 2025, and where in the city?
2. Which sectors hire the most, and which hire young people and graduates (QWI)?
3. What is the direction right now (CES through Aug 2026) and in output terms (GDP 2019 → 2024)?
4. Which sectors hold the degree-reachable jobs?

**Rules (plan §3d):** 2019 is the base year everywhere; 2020 is shown, never a base. Published series only; the one derived series (all-ownership employment by sector with government filed per D18) is labeled as the D8 input.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 58, "display.width", 240,
              "display.float_format", "{:,.1f}".format)


def md(s: str):
    display(Markdown(s))


def show(df, title=None):
    if title:
        md(f"**{title}**")
    display(df)


ind = q("select industry_key, level, industry_title, naics_sector from dim_industry").set_index("industry_key")
title = ind.industry_title
SECTORS = ind[ind.level == "sector"].index.tolist()

## 1. NYC employment by sector, 2019 → 2025 (QCEW)

NYS DOL publishes the NYC private sector completely (22 keys: 20 sectors, unclassified `99`, total `00`), but all-ownership rows only for the sectors where government needs no suppression (10 of 20 at NYC level). So the published sector series is private employment; government is one published total beside it. The all-ownership view by sector exists only as the D8 input (`audit_nyc_employment_by_pattern_sector`, government filed per D18), shown second and labeled.

In [2]:
qcew = q("""
select year, area_key, ownership, industry_key, employment, establishments, avg_annual_pay, disclosed, source_key
from fct_industry_employment_annual
""")
print(f"fct_industry_employment_annual rows: {len(qcew):,}")
nyc_priv = qcew[(qcew.area_key == "nyc") & (qcew.ownership == "Private")]
P = nyc_priv.pivot_table(index="industry_key", columns="year", values="employment")
P.index = [f"{k} {title.get(k, '')}" for k in P.index]
priv = pd.DataFrame({"2019": P[2019], "2020": P[2020], "2024": P[2024], "2025": P[2025],
                     "change 19→25": P[2025] - P[2019], "growth %": (P[2025] / P[2019] - 1) * 100,
                     "share 2025 %": P[2025] / P.loc["00 Total, all industries", 2025] * 100}).sort_values("2025", ascending=False)
show(priv.round(1), "NYC private employment by sector (QCEW annual average)")

gov = qcew[(qcew.area_key == "nyc") & (qcew.industry_key == "00")].pivot_table(index="ownership", columns="year", values="employment")
show(pd.DataFrame({"2019": gov[2019], "2025": gov[2025], "change": gov[2025] - gov[2019], "growth %": (gov[2025] / gov[2019] - 1) * 100}).round(1),
     "NYC totals by ownership")

fct_industry_employment_annual rows: 2,241


**NYC private employment by sector (QCEW annual average)**

,2019,2020,2024,2025,change 19→25,growth %,share 2025 %
"00 Total, all industries","3,916,345.0","3,421,539.0","4,063,618.0","4,074,085.0","157,740.0",4.0,100.0
62 Health Care and Social Assistance,"791,836.0","762,055.0","984,503.0","998,530.0","206,694.0",26.1,24.5
"54 Professional, Scientific, and Technical Services","413,508.0","404,398.0","437,038.0","431,275.0","17,767.0",4.3,10.6
52 Finance and Insurance,"339,267.0","333,798.0","352,445.0","361,274.0","22,007.0",6.5,8.9
72 Accommodation and Food Services,"370,556.0","216,375.0","354,305.0","358,673.0","-11,883.0",-3.2,8.8
44-45 Retail Trade,"344,589.0","284,600.0","294,934.0","294,622.0","-49,967.0",-14.5,7.2
56 Administrative and Support and Waste Management and Remediation Services,"273,575.0","211,109.0","240,892.0","237,268.0","-36,307.0",-13.3,5.8
51 Information,"203,821.0","201,486.0","216,299.0","210,768.0","6,947.0",3.4,5.2
61 Educational Services,"186,758.0","174,943.0","197,363.0","201,260.0","14,502.0",7.8,4.9
48-49 Transportation and Warehousing,"130,801.0","111,742.0","173,358.0","170,937.0","40,136.0",30.7,4.2


**NYC totals by ownership**

,2019,2025,change,growth %
ownership,,,,
Federal Government,"48,341.0","44,437.0","-3,904.0",-8.1
Local Government,"492,563.0","498,650.0","6,087.0",1.2
Private,"3,916,345.0","4,074,085.0","157,740.0",4.0
State Government,"35,459.0","35,756.0",297.0,0.8
"Total, All Government","576,363.0","578,843.0","2,480.0",0.4
"Total, All Ownerships","4,492,708.0","4,652,929.0","160,221.0",3.6


In [3]:
d8 = q("""
select year, pattern_sector, employment, private_employment, government_disclosed, government_from_nysdol, government_undisclosed_allocated
from audit_nyc_employment_by_pattern_sector
""")
A = d8.pivot_table(index="pattern_sector", columns="year", values="employment")
A.index = [f"{k} {'Government (OEWS 99 pattern, D18)' if k == '99' else title.get(k, '')}" for k in A.index]
show(pd.DataFrame({"2019": A[2019], "2025": A[2025], "change": A[2025] - A[2019], "growth %": (A[2025] / A[2019] - 1) * 100})
       .sort_values("2025", ascending=False).round(1),
     "All ownerships by pattern sector, government filed per D18 (D8 input, derived, sums to the QCEW total)")

**All ownerships by pattern sector, government filed per D18 (D8 input, derived, sums to the QCEW total)**

,2019,2025,change,growth %
62 Health Care and Social Assistance,"834,487.9","1,054,936.3","220,448.4",26.4
"54 Professional, Scientific, and Technical Services","414,685.6","433,780.2","19,094.6",4.6
52 Finance and Insurance,"340,233.1","363,372.6","23,139.4",6.8
"99 Government (OEWS 99 pattern, D18)","370,619.0","360,757.0","-9,862.0",-2.7
72 Accommodation and Food Services,"371,611.2","360,756.5","-10,854.8",-2.9
61 Educational Services,"334,824.8","354,371.1","19,546.2",5.8
44-45 Retail Trade,"345,570.3","296,333.4","-49,236.9",-14.2
56 Administrative and Support and Waste Management and Remediation Services,"274,354.1","238,646.2","-35,707.8",-13.0
51 Information,"204,401.4","211,992.3","7,590.9",3.7
48-49 Transportation and Warehousing,"148,987.5","187,466.9","38,479.5",25.8


## 2. Where in the city: boroughs
Private employment 2019 → 2025 by borough, and each sector's growth by borough.

In [4]:
BORO = ["manhattan", "brooklyn", "queens", "bronx", "staten_island"]
b = qcew[(qcew.area_key.isin(BORO)) & (qcew.ownership == "Private")]
bt = b[b.industry_key == "00"].pivot_table(index="area_key", columns="year", values="employment").loc[BORO]
show(pd.DataFrame({"2019": bt[2019], "2025": bt[2025], "change": bt[2025] - bt[2019], "growth %": (bt[2025] / bt[2019] - 1) * 100}).round(1),
     "Private employment by borough")
bs = b[b.industry_key.isin(SECTORS)].pivot_table(index="industry_key", columns=["area_key", "year"], values="employment")
grow = pd.DataFrame({a: (bs[(a, 2025)] / bs[(a, 2019)] - 1) * 100 for a in BORO})
grow.index = [f"{k} {title[k]}" for k in grow.index]
show(grow.round(1), "Sector growth 2019 → 2025 by borough, % (blank = not disclosed in one of the years)")

**Private employment by borough**

,2019,2025,change,growth %
area_key,,,,
manhattan,"2,295,573.0","2,255,712.0","-39,861.0",-1.7
brooklyn,"672,162.0","801,460.0","129,298.0",19.2
queens,"595,330.0","628,098.0","32,768.0",5.5
bronx,"248,801.0","276,559.0","27,758.0",11.2
staten_island,"104,480.0","112,256.0","7,776.0",7.4


**Sector growth 2019 → 2025 by borough, % (blank = not disclosed in one of the years)**

,manhattan,brooklyn,queens,bronx,staten_island
"11 Agriculture, Forestry, Fishing and Hunting",-15.4,16.6,288.5,138.1,NaN
"21 Mining, Quarrying, and Oil and Gas Extraction",86.7,NaN,-2.6,NaN,NaN
22 Utilities,NaN,9.1,NaN,NaN,-22.9
23 Construction,-18.3,-16.2,-11.9,-15.9,-9.7
31-33 Manufacturing,-32.8,-12.0,-22.5,-18.7,-3.4
42 Wholesale Trade,-5.7,-8.3,-9.9,-3.9,-12.4
44-45 Retail Trade,-21.3,-6.5,-10.7,-8.9,-12.5
48-49 Transportation and Warehousing,101.7,40.2,4.6,52.2,46.4
51 Information,3.2,21.7,-5.5,-7.0,-42.1
52 Finance and Insurance,10.1,-21.0,-22.1,-41.2,21.2


## 3. Hiring: who hires, and who hires young people and graduates (QWI)

QWI counts new hires (people who did not work for that employer in the previous year) every quarter, by sector, with the worker's age and (for workers 25+) education. 2025 Q4 is not yet released, so annual comparisons stop at 2024; 2025 is shown Q1–Q3 against the same quarters of 2024. NYC = sum of the five boroughs (QWI publishes counties).

In [5]:
qwi = q("""
select year, quarter, area_key, industry_key, breakdown, group_code, emp_begin, hires_all, hires_new, separations, quarter_complete
from fct_industry_hiring_quarterly
""")
print(f"fct_industry_hiring_quarterly rows: {len(qwi):,}")
grp = q("select breakdown, group_code, group_label from ref.qwi_group").set_index(["breakdown", "group_code"]).group_label
qwi["group_label"] = [grp.get((b_, g)) for b_, g in zip(qwi.breakdown, qwi.group_code)]

tot = (qwi[(qwi.breakdown == "age") & (qwi.group_code == "A00") & qwi.quarter_complete & (qwi.industry_key != "92")]
         .groupby(["year", "quarter", "industry_key"], as_index=False)[["hires_new", "hires_all", "separations", "emp_begin"]].sum())
ann = tot.groupby(["year", "industry_key"]).agg(new_hires=("hires_new", "sum"), all_hires=("hires_all", "sum"),
                                                 separations=("separations", "sum"), emp=("emp_begin", "mean"), quarters=("quarter", "nunique")).reset_index()
full = ann[ann.quarters == 4]
H = full.pivot_table(index="industry_key", columns="year", values="new_hires")
E = full.pivot_table(index="industry_key", columns="year", values="emp")
hire = pd.DataFrame({"new hires 2019": H[2019], "new hires 2024": H[2024], "growth %": (H[2024] / H[2019] - 1) * 100,
                     "hiring rate 2024 %": H[2024] / E[2024] * 100}).sort_values("new hires 2024", ascending=False)
hire.index = [f"{k} {title.get(k, '')}" for k in hire.index]
show(hire.round(1), "New hires a year by sector, NYC (sum of boroughs); hiring rate = new hires ÷ average employment")
h25 = tot[(tot.year.isin([2024, 2025])) & (tot.quarter <= 3)].groupby(["year", "industry_key"]).hires_new.sum().unstack(0)
h25["change %"] = (h25[2025] / h25[2024] - 1) * 100
h25.index = [f"{k} {title.get(k, '')}" for k in h25.index]
show(h25.sort_values(2025, ascending=False).round(1), "New hires Q1–Q3, 2025 vs 2024")

fct_industry_hiring_quarterly rows: 39,648


**New hires a year by sector, NYC (sum of boroughs); hiring rate = new hires ÷ average employment**

,new hires 2019,new hires 2024,growth %,hiring rate 2024 %
"00 Total, all industries","2,284,767.0","2,232,885.0",-2.3,52.9
62 Health Care and Social Assistance,"362,117.0","458,684.0",26.7,46.4
72 Accommodation and Food Services,"313,636.0","269,206.0",-14.2,77.1
48-49 Transportation and Warehousing,"82,265.0","241,110.0",193.1,120.0
56 Administrative and Support and Waste Management and Remediation Services,"255,544.0","213,046.0",-16.6,80.5
"54 Professional, Scientific, and Technical Services","208,978.0","180,687.0",-13.5,40.7
44-45 Retail Trade,"214,328.0","155,562.0",-27.4,52.0
51 Information,"216,210.0","147,773.0",-31.7,58.2
52 Finance and Insurance,"100,253.0","101,474.0",1.2,27.2
61 Educational Services,"89,779.0","93,643.0",4.3,42.1


**New hires Q1–Q3, 2025 vs 2024**

year,2024,2025,change %
"00 Total, all industries","1,701,814.0","1,733,833.0",1.9
62 Health Care and Social Assistance,"345,711.0","497,388.0",43.9
72 Accommodation and Food Services,"201,744.0","203,951.0",1.1
56 Administrative and Support and Waste Management and Remediation Services,"158,076.0","155,060.0",-1.9
"54 Professional, Scientific, and Technical Services","139,410.0","136,832.0",-1.8
44-45 Retail Trade,"107,827.0","108,632.0",0.7
51 Information,"115,277.0","103,220.0",-10.5
48-49 Transportation and Warehousing,"196,539.0","95,461.0",-51.4
52 Finance and Insurance,"79,824.0","85,245.0",6.8
61 Educational Services,"70,089.0","69,579.0",-0.7


A series break to flag before reading the hiring table. Transportation and Warehousing (`48-49`) shows 193% more new hires in 2024 than 2019 and a 51% drop in 2025. The quarterly series below shows why: one quarter, 2024 Q1, with 128 k "new hires" (a normal quarter is 15–45 k), followed by employment stepping from 142 k to 222 k. That is an employer reclassification into the sector, not hiring: workers look new to an employer that changed its industry code. QCEW's +31% for the sector 2019 → 2025 and CES's +2% for the same sector are the same break seen from two sides.

In [6]:
tw = tot[tot.industry_key == "48-49"].pivot_table(index="year", columns="quarter", values=["hires_new", "emp_begin"])
show(tw.round(0), "Transportation and Warehousing, NYC: new hires and beginning-of-quarter employment by quarter (2025 Q4 not yet released)")

**Transportation and Warehousing, NYC: new hires and beginning-of-quarter employment by quarter (2025 Q4 not yet released)**

emp_begin                               hires_new                           
quarter         1         2         3         4         1        2        3        4
year                                                                                
2019    136,126.0 130,903.0 133,900.0 135,588.0  11,759.0 18,024.0 20,277.0 32,205.0
2020    141,676.0 118,667.0 111,990.0 109,659.0  15,356.0 19,344.0 18,791.0 32,261.0
2021    126,991.0 120,095.0 123,625.0 127,670.0  11,480.0 18,673.0 25,224.0 42,455.0
2022    138,731.0 135,057.0 136,189.0 136,558.0  20,490.0 20,605.0 23,598.0 28,553.0
2023    140,130.0 135,062.0 136,646.0 136,706.0  14,568.0 18,212.0 18,307.0 28,157.0
2024    141,812.0 221,649.0 220,505.0 219,986.0 128,413.0 35,051.0 33,075.0 44,571.0
2025    230,191.0 226,149.0 222,948.0       NaN  32,278.0 32,774.0 30,409.0      NaN

In [7]:
age = qwi[(qwi.breakdown == "age") & (qwi.year == 2024) & qwi.quarter_complete & (qwi.industry_key != "92")]
ag = age.groupby(["industry_key", "group_code"]).hires_new.sum().unstack()
young = pd.DataFrame({"new hires 2024": ag["A00"], "share 22-24 %": ag["A03"] / ag["A00"] * 100,
                      "share 25-34 %": ag["A04"] / ag["A00"] * 100, "share under 25 %": (ag["A01"] + ag["A02"] + ag["A03"]) / ag["A00"] * 100})
edu = qwi[(qwi.breakdown == "education") & (qwi.year == 2024) & qwi.quarter_complete & (qwi.industry_key != "92")]
ed = edu.groupby(["industry_key", "group_code"]).hires_new.sum().unstack()
ed25 = ed[["E1", "E2", "E3", "E4"]].sum(axis=1)
young["share bachelor's+ among 25+ hires %"] = ed["E4"] / ed25 * 100
young["share some college/associate among 25+ hires %"] = ed["E3"] / ed25 * 100
young.index = [f"{k} {title.get(k, '')}" for k in young.index]
show(young.sort_values("new hires 2024", ascending=False).round(1), "Who each sector hired in 2024: age and education shares of new hires")

**Who each sector hired in 2024: age and education shares of new hires**

,new hires 2024,share 22-24 %,share 25-34 %,share under 25 %,share bachelor's+ among 25+ hires %,share some college/associate among 25+ hires %
"00 Total, all industries","2,232,885.0",10.6,29.9,22.3,30.9,28.1
62 Health Care and Social Assistance,"458,684.0",7.9,25.3,16.2,28.0,27.6
72 Accommodation and Food Services,"269,206.0",11.6,29.7,28.3,28.4,28.1
48-49 Transportation and Warehousing,"241,110.0",9.0,31.6,17.2,27.6,28.0
56 Administrative and Support and Waste Management and Remediation Services,"213,046.0",10.7,31.4,19.5,26.7,30.1
"54 Professional, Scientific, and Technical Services","180,687.0",13.8,36.1,23.4,39.6,27.4
44-45 Retail Trade,"155,562.0",14.1,26.6,41.3,28.8,28.7
51 Information,"147,773.0",8.8,34.2,14.5,37.7,28.2
52 Finance and Insurance,"101,474.0",12.8,33.0,24.8,43.7,25.7
61 Educational Services,"93,643.0",15.8,31.3,35.7,38.0,27.5


## 4. The direction now: CES monthly through Aug 2026
Latest 12 months (Sep 2025 – Aug 2026) against the 12 before, and against the 2019 average. CES is a survey of establishments benchmarked to QCEW; its NYC total runs 3.5–5.2% above QCEW (step 4 test), so levels differ, directions agree.

In [8]:
ces = q("""
select year, month, series_code, series_title, industry_key, ces_level, employment
from fct_industry_employment_monthly where ces_level in ('aggregate', 'supersector')
""")
ces["date"] = pd.to_datetime(dict(year=ces.year, month=ces.month, day=1))
last = ces.date.max()
w1 = ces[ces.date > last - pd.DateOffset(months=12)]
w0 = ces[(ces.date <= last - pd.DateOffset(months=12)) & (ces.date > last - pd.DateOffset(months=24))]
base = ces[ces.year == 2019]
C = pd.DataFrame({"2019 avg": base.groupby("series_title").employment.mean(),
                  "prior 12 mo": w0.groupby("series_title").employment.mean(),
                  "latest 12 mo": w1.groupby("series_title").employment.mean()})
C["vs prior %"] = (C["latest 12 mo"] / C["prior 12 mo"] - 1) * 100
C["vs 2019 %"] = (C["latest 12 mo"] / C["2019 avg"] - 1) * 100
print(f"latest month: {last:%b %Y}; windows of {w1.date.nunique()} and {w0.date.nunique()} months")
show(C.sort_values("latest 12 mo", ascending=False).round(1), "CES NYC, average monthly employment")

latest month: Aug 2026; windows of 12 and 12 months


**CES NYC, average monthly employment**

,2019 avg,prior 12 mo,latest 12 mo,vs prior %,vs 2019 %
series_title,,,,,
Total Nonfarm,"4,661,383.3","4,823,858.3","4,834,133.3",0.2,3.7
Service-Providing,"4,431,966.7","4,631,091.7","4,646,900.0",0.3,4.8
Total Private,"4,063,016.7","4,213,183.3","4,210,591.7",-0.1,3.6
Private Service Providing,"3,833,600.0","4,020,416.7","4,023,358.3",0.1,4.9
Private Education and Health Services,"1,055,366.7","1,279,916.7","1,280,291.7",0.0,21.3
Professional and Business Services,"772,341.7","796,166.7","799,758.3",0.5,3.5
Government,"598,366.7","610,675.0","623,541.7",2.1,4.2
"Trade, Transportation, and Utilities","636,441.7","582,775.0","580,950.0",-0.3,-8.7
Financial Activities,"485,125.0","511,766.7","518,891.7",1.4,7.0


## 5. Output: GDP by sector, 2019 → 2024 (BEA county GDP)
Boroughs only (no NYC line); NYC = sum of boroughs in current dollars. Shares divide by BEA's all-industry line (line 1, summed over boroughs), the same basis as `mart.industry_profile.gdp_share`; the sum of the sector lines is about 1% short of it because some borough cells are withheld. Real growth is read per sector from the chained-2017 series summed over boroughs; BEA warns that chained-dollar sums are approximate, so the real column is a direction, not an accounting identity.

In [9]:
gdp = q("""
select year, area_key, line_code, line_title, industry_key, gdp_current_thousands, gdp_real_chained2017_thousands, is_withheld
from fct_industry_gdp
""")
print(f"fct_industry_gdp rows: {len(gdp):,}; withheld cells: {int(gdp.is_withheld.sum())}")
g = gdp[gdp.industry_key.notna() & (gdp.line_code.astype(int) > 2)]
cur = g.pivot_table(index="line_title", columns="year", values="gdp_current_thousands", aggfunc="sum")
real = g.pivot_table(index="line_title", columns="year", values="gdp_real_chained2017_thousands", aggfunc="sum")
all_ind = gdp[gdp.line_code.astype(str) == "1"].groupby("year").gdp_current_thousands.sum()
print(f"2024 all-industry line {all_ind[2024] / 1e6:,.1f} $B vs sum of sector lines {cur[2024].sum() / 1e6:,.1f} $B (withheld cells)")
G = pd.DataFrame({"GDP 2024 $B": cur[2024] / 1e6, "share 2024 %": cur[2024] / all_ind[2024] * 100,
                  "real growth 19→24 %": (real[2024] / real[2019] - 1) * 100,
                  "nominal growth 19→24 %": (cur[2024] / cur[2019] - 1) * 100}).sort_values("GDP 2024 $B", ascending=False)
show(G.round(1), "NYC GDP by sector (sum of boroughs)")
man = gdp[(gdp.line_code.astype(str) == "1")].pivot_table(index="area_key", columns="year", values="gdp_current_thousands")
show(pd.DataFrame({"2019 $B": man[2019] / 1e6, "2024 $B": man[2024] / 1e6, "share 2024 %": man[2024] / man[2024].sum() * 100}).round(1), "All-industry GDP by borough")

fct_industry_gdp rows: 660; withheld cells: 36
2024 all-industry line 1,377.8 $B vs sum of sector lines 1,363.7 $B (withheld cells)


**NYC GDP by sector (sum of boroughs)**

,GDP 2024 $B,share 2024 %,real growth 19→24 %,nominal growth 19→24 %
line_title,,,,
Finance and insurance,362.2,26.3,10.8,41.4
Information,173.9,12.6,36.9,32.2
Real estate and rental and leasing,160.7,11.7,1.3,23.1
"Professional, scientific, and technical services",144.1,10.5,18.0,31.3
Government and government enterprises,95.3,6.9,-4.7,14.6
Health care and social assistance,88.4,6.4,22.4,43.4
Retail trade,46.6,3.4,2.0,25.9
Wholesale trade,44.8,3.3,-18.0,12.5
Accommodation and food services,41.0,3.0,-12.0,19.2


**All-industry GDP by borough**

,2019 $B,2024 $B,share 2024 %
area_key,,,
bronx,47.1,58.3,4.2
brooklyn,108.5,145.9,10.6
manhattan,780.0,"1,006.7",73.1
queens,111.5,143.1,10.4
staten_island,17.7,23.8,1.7


## 6. Which sectors hold the degree-reachable jobs?
`bridge_industry_job` is the OEWS research staffing pattern for New York (D3): each sector's employment split by job. Combined with `dim_job.is_degree_entry` it says what share of a sector's workforce is in degree-reachable jobs, and which those jobs are.

In [10]:
bij = q("""
select b.industry_key, b.soc_code, j.soc_title, b.share_of_industry, b.emp_ny_2025, j.is_degree_entry, j.is_all_other
from bridge_industry_job b join dim_job j using (soc_code)
""")
print(f"bridge_industry_job rows: {len(bij):,}")
deg = bij.groupby("industry_key").apply(lambda d: pd.Series({
    "pattern coverage": d.share_of_industry.sum(),
    "degree-reachable share %": d.loc[d.is_degree_entry, "share_of_industry"].sum() / d.share_of_industry.sum() * 100,
    "top degree jobs": ", ".join(d[d.is_degree_entry & ~d.is_all_other].nlargest(3, "emp_ny_2025").soc_title.str.slice(0, 32))}), include_groups=False)
nyc25 = P[2025].rename(index=lambda s: s.split(" ")[0])
deg["NYC private jobs 2025"] = deg.index.map(nyc25)
deg["≈ degree-reachable NYC jobs"] = deg["NYC private jobs 2025"] * deg["degree-reachable share %"] / 100
deg.index = [f"{k} {title.get(k, '')}" for k in deg.index]
show(deg.sort_values("≈ degree-reachable NYC jobs", ascending=False).round(1), "Degree-reachable share of each sector's workforce (NY pattern) and the sector's largest degree jobs")

bridge_industry_job rows: 7,438


**Degree-reachable share of each sector's workforce (NY pattern) and the sector's largest degree jobs**

,pattern coverage,degree-reachable share %,top degree jobs,NYC private jobs 2025,≈ degree-reachable NYC jobs
"54 Professional, Scientific, and Technical Services",1.0,64.2,"Software Developers, Accountants and Auditors, Managem...","431,275.0","276,701.4"
52 Finance and Insurance,1.0,69.5,"Securities, Commodities, and Fin, Financial Managers, ...","361,274.0","251,185.7"
62 Health Care and Social Assistance,1.0,20.7,"Registered Nurses, Medical and Health Services Mana, P...","998,530.0","207,091.2"
51 Information,1.0,64.7,"Software Developers, Producers and Directors, Editors","210,768.0","136,379.8"
61 Educational Services,1.0,42.8,"Elementary School Teachers, Exce, Secondary School Tea...","201,260.0","86,222.3"
55 Management of Companies and Enterprises,0.9,71.1,"General and Operations Managers, Financial Managers, M...","78,524.0","55,806.1"
56 Administrative and Support and Waste Management and Remediation Services,1.0,21.3,"Registered Nurses, General and Operations Managers, Hu...","237,268.0","50,628.5"
81 Other Services (except Public Administration),1.0,25.8,"General and Operations Managers, Clergy, Labor Relatio...","162,920.0","42,072.1"
42 Wholesale Trade,1.0,29.0,"General and Operations Managers, Sales Representatives...","122,714.0","35,579.0"
44-45 Retail Trade,1.0,7.8,"General and Operations Managers, Buyers and Purchasing...","294,622.0","23,038.8"


## 7. Checks

In [11]:
chk = []
pt = nyc_priv.pivot_table(index="year", columns="industry_key", values="employment")
sec_sum = pt[[c for c in pt.columns if c in SECTORS or c == "99"]].sum(axis=1)
chk.append(["QCEW NYC private: Σ sectors (+99) vs private total, max |gap|", f"{(sec_sum - pt['00']).abs().max():,.0f} jobs"])
tot_rows = qcew[(qcew.area_key == "nyc") & (qcew.industry_key == "00")].pivot_table(index="year", columns="ownership", values="employment")
chk.append(["QCEW NYC: private + government vs all ownerships, max |gap|",
            f"{(tot_rows['Private'] + tot_rows['Total, All Government'] - tot_rows['Total, All Ownerships']).abs().max():,.0f} jobs"])
ces_tot = ces[ces.series_code == "00000000"].groupby("year").employment.mean()
ratio = (ces_tot / tot_rows["Total, All Ownerships"]).dropna()
chk.append(["CES annual mean ÷ QCEW all-ownership total, 2019–2025", f"{ratio.min():.3f} – {ratio.max():.3f}"])
a24 = qwi[(qwi.breakdown == "age") & (qwi.year == 2024) & (qwi.industry_key == "00")].groupby("group_code").hires_new.sum()
chk.append(["QWI 2024 NYC new hires: Σ age groups vs A00", f"{a24.drop('A00').sum():,.0f} vs {a24['A00']:,.0f}"])
m24 = gdp[(gdp.area_key == "manhattan") & (gdp.year == 2024)]
chk.append(["GDP Manhattan 2024: Σ sector lines vs line 1",
            f"{m24[m24.industry_key.notna() & (m24.line_code.astype(int) > 2)].gdp_current_thousands.sum() / 1e6:,.1f} vs {m24[m24.line_code.astype(str) == '1'].gdp_current_thousands.iloc[0] / 1e6:,.1f} $B"])
t62 = {
    "QCEW NYC private 62, 2024": nyc_priv[(nyc_priv.year == 2024) & (nyc_priv.industry_key == "62")].employment.iloc[0],
    "QWI NYC new hires 62, 2024": full[(full.year == 2024) & (full.industry_key == "62")].new_hires.iloc[0],
    "D8 input 62, 2024 (all ownerships)": d8[(d8.year == 2024) & (d8.pattern_sector == "62")].employment.iloc[0],
    "GDP Manhattan 62, 2024 ($B)": m24[m24.industry_key == "62"].gdp_current_thousands.iloc[0] / 1e6,
}
for k, v in t62.items():
    chk.append([f"Trace Health Care · {k}", f"{v:,.1f}"])
show(pd.DataFrame(chk, columns=["check", "result"]), "Checks (expected: gaps ≤ 3 jobs; CES ratio 1.035–1.052; QWI within noise; GDP Σ = line 1 ± 0.1%; trace = step 4 §9 values 984,503 / 458,684 / 1,037,428 (= 1,031,152 + 6,276 state-62 fill added in the §9 refinement) / 34.7)")

**Checks (expected: gaps ≤ 3 jobs; CES ratio 1.035–1.052; QWI within noise; GDP Σ = line 1 ± 0.1%; trace = step 4 §9 values 984,503 / 458,684 / 1,037,428 (= 1,031,152 + 6,276 state-62 fill added in the §9 refinement) / 34.7)**

,check,result
0,"QCEW NYC private: Σ sectors (+99) vs private total, ma...",3 jobs
1,"QCEW NYC: private + government vs all ownerships, max ...",1 jobs
2,"CES annual mean ÷ QCEW all-ownership total, 2019–2025",1.035 – 1.051
3,QWI 2024 NYC new hires: Σ age groups vs A00,"2,232,994 vs 2,232,885"
4,GDP Manhattan 2024: Σ sector lines vs line 1,"1,006.7 vs 1,006.7 $B"
5,"Trace Health Care · QCEW NYC private 62, 2024","984,503.0"
6,"Trace Health Care · QWI NYC new hires 62, 2024","458,684.0"
7,"Trace Health Care · D8 input 62, 2024 (all ownerships)","1,037,428.5"
8,"Trace Health Care · GDP Manhattan 62, 2024 ($B)",34.7


## 8. Findings (from the tables above)

1. NYC's job growth since 2019 is health care, and almost nothing else. Private employment +158 k (+4.0%) 2019 → 2025; Health Care and Social Assistance alone +207 k (+26%), now a quarter of all private jobs. Transportation and Warehousing +40 k (+31%) is the only other large gain, and §3 shows it is mostly an employer reclassification in 2024 Q1. Finance +22 k (+6.5%), Professional and Technical +18 k (+4.3%), Education +15 k (+8%), Information +7 k (+3%). Losing: Retail −50 k (−14.5%), Administrative and Support −36 k (−13%), Construction −23 k (−15%), Manufacturing −15 k (−22%), Other Services −15 k, Accommodation and Food −12 k (−3%, still below 2019). Government flat (+2 k).
2. **The gain is outside Manhattan.** Manhattan private jobs −40 k (−1.7%); Brooklyn +129 k (+19%), Bronx +28 k (+11%), Queens +33 k (+6%), Staten Island +8 k (+7%). Health care grew 42% in Brooklyn and 25% in Queens and the Bronx against 11% in Manhattan; Professional and Technical +34% in Brooklyn against +2.5% in Manhattan. Finance is a Manhattan story (+10%) and shrank in the other boroughs.
3. Hiring volume is flat and churn-driven. 2.23 M new hires in 2024, −2% vs 2019, on 4.2 M private jobs (a 53% annual new-hire rate). Health care hires the most (459 k, +27% vs 2019) and is up 44% in 2025 Q1–Q3; Information (−32%), Retail (−27%), Construction (−27%) and Professional and Technical (−14%) hire far less than in 2019. High-churn sectors (Accommodation 77%, Administrative 81%, Arts 79%) are where volume comes from; Finance (27%), Real Estate (30%) and Wholesale (32%) hire rarely.
4. Who hires graduates and young workers. Among hires aged 25+, the bachelor's-or-higher share is 44% in Finance, 40% in Professional and Technical, 38% in Information and Education, against 31% overall and 28% in Health Care. Workers under 25 are 41% of Retail hires, 36% of Education, 28% of Accommodation and Food, 23% overall. The 22–24 band (new bachelor's graduates) is largest in Education (16%), Retail (14%), Professional and Technical (14%) and Finance (13%). QWI education is imputed and covers workers 25+ only; "some college or associate" cannot be split from "some college".
5. Right now (CES to Aug 2026): flat. Total nonfarm +0.2% over the prior 12 months, +3.7% vs 2019. Private Education and Health +21% vs 2019 and no longer growing (0.0% over the year); Financial Activities +1.4% on the year; Government +2.1%; Goods-producing −2.9%, Manufacturing −4.4%, Leisure and Hospitality −1.2%. Retail and Manufacturing sit 14% and 25% below 2019.
6. Output and jobs point in different directions. Finance and Insurance is 26.3% of NYC GDP in 2024 ($362 B of $1,378 B) with 8.9% of private jobs; Health Care is 6.4% of GDP with 24.5% of jobs (shares of BEA's all-industry line; an earlier draft divided by the sum of sector lines, which withheld borough cells leave 1% short, and read 27% / 6.5%). Real growth 2019 → 2024: Transportation +46% (the reclassification again), Information +37%, Health Care +22%, Professional and Technical +18%, Finance +11%; Construction −22%, Wholesale −18%, Other Services −18%, Administrative −13%, Accommodation −12%. Manhattan is 73% of NYC output.
7. Where the degree-reachable jobs live (NY staffing pattern × NYC private jobs). Professional and Technical ≈ 277 k degree-reachable jobs (64% of the sector), Finance ≈ 251 k (70%), Health Care ≈ 207 k (21% of a very large sector), Information ≈ 136 k (65%), Education ≈ 86 k (43%), Management of Companies ≈ 56 k (71%). Retail, Accommodation and Food, Transportation and Construction are big sectors with 3–15% degree-reachable shares. So the sector growing fastest (health) is the one where degree-reachable jobs are a minority of the work, and the sectors richest in degree jobs (professional, finance, information) grew 3–7%.
8. **Checks.** QCEW private sectors sum to the private total and private + government to the all-ownership total within 3 jobs every year; CES runs 3.5–5.2% above QCEW; QWI age groups sum to the total within noise; Manhattan GDP sector lines sum to the total within 0.1%; the Health Care trace matches step 4 §9 (984,503 / 458,684 / 1,031,152 / $34.7 B).
9. **Limits.** QCEW counts jobs at establishments, not residents (Manhattan's decline is partly remote work); QWI is private-sector only, noise-infused, and 2025 Q4 is missing; education is imputed; CES is a survey; GDP is by borough with 36 withheld cells and chained-dollar sums are approximate; the 48-49 reclassification is unexplained and needs a note before any page quotes transportation growth.
10. **For the mart.** Industry pages should lead with private employment by sector and borough (published), hiring volume and the graduate share of hires (QWI 2024), and CES for the current direction; label the all-ownership-by-sector series as the D8 estimate; footnote the 48-49 break and the health-care GDP/jobs contrast.